# Data Modelling — Regression

*Generated by DoML — Do Machine Learning*

> **Note:** This notebook uses Python regardless of the `analysis_language` config setting.
> scikit-learn, SHAP, and Optuna do not have comparable R equivalents.

In [ ]:
# Reproducibility — REPR-01 (non-negotiable per CLAUDE.md)
import os, random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import json, warnings
from pathlib import Path
from datetime import date

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')

In [ ]:
# Project root — REPR-02 (non-negotiable per CLAUDE.md)
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', '/home/jovyan/work'))

with open(PROJECT_ROOT / '.planning' / 'config.json') as f:
    config = json.load(f)

problem_type = config.get('problem_type', 'regression')
TARGET = config.get('target_column')
row_count_threshold = 2000  # recommend 10-fold CV below this size

if problem_type != 'regression':
    raise ValueError(f"This notebook is for regression problems. problem_type='{problem_type}' found in config.json.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Problem type: {problem_type}")
print(f"Target column: {TARGET}")

In [ ]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
preprocessed_files = (list(processed_dir.glob('preprocessed_*.csv')) +
                      list(processed_dir.glob('preprocessed_*.parquet')))

if not preprocessed_files:
    raise FileNotFoundError(
        "No preprocessed dataset found. Run the preprocessing notebook first."
    )

input_file = preprocessed_files[0]
print(f"Loading: {input_file.name}")

if input_file.suffix == '.parquet':
    df = pd.read_parquet(input_file)
else:
    df = pd.read_csv(input_file)

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found. Check config.json target_column.")

X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Target: {TARGET}  (mean={y.mean():.3f}, std={y.std():.3f})")

## Step 1: Holdout Split

80% training / 20% test. The test set is **sealed** until Step 5 (final model evaluation). No model selection decisions will be made using test set metrics — only CV metrics inform model selection.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

print(f"Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows")

# Recommend 10-fold CV for small datasets
n_folds = 5
if len(X_train) < row_count_threshold:
    print(f"\n⚠️  Small training set ({len(X_train):,} rows < {row_count_threshold}). "
          f"Consider using 10-fold CV by setting n_folds = 10 below.")
print(f"Using {n_folds}-fold CV.")

## Step 2: Model Definitions

Models evaluated (in leaderboard order):
- **DummyRegressor(mean)** — baseline; every model must beat this (MOD-07)
- **LinearRegression** — no regularization
- **Ridge(alpha=1.0)** — L2 regularization
- **Lasso(alpha=0.1)** — L1 regularization (feature selection)
- **RandomForestRegressor** — ensemble, 100 trees
- **XGBRegressor** — gradient boosted trees (XGBoost)
- **LGBMRegressor** — gradient boosted trees (LightGBM)

In [ ]:
models = {
    'DummyRegressor(mean)':    DummyRegressor(strategy='mean'),
    'LinearRegression':         LinearRegression(),
    'Ridge':                    Ridge(alpha=1.0, random_state=SEED),
    'Lasso':                    Lasso(alpha=0.1, random_state=SEED, max_iter=10000),
    'RandomForestRegressor':    RandomForestRegressor(n_estimators=100, random_state=SEED),
    'XGBRegressor':             xgb.XGBRegressor(n_estimators=100, random_state=SEED, verbosity=0),
    'LGBMRegressor':            lgb.LGBMRegressor(n_estimators=100, random_state=SEED, verbose=-1),
}

## Step 3: Cross-Validated Leaderboard

5-fold KFold CV on the training set. Metrics shown are mean ± std across folds.

**Primary metric:** RMSE (lower is better)
**Secondary metrics:** MAE, R²

The test set is NOT used here — only CV scores inform model selection (MOD-06).

In [ ]:
kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
leaderboard_rows = []

# Store fitted models and first-split data for SHAP
fitted_models = {}
shap_data = {}

for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train, cv=kf,
        scoring={
            'rmse':  'neg_root_mean_squared_error',
            'mae':   'neg_mean_absolute_error',
            'r2':    'r2',
        },
        return_estimator=True,
        return_train_score=False,
    )
    
    rmse_mean = -scores['test_rmse'].mean()
    rmse_std  = scores['test_rmse'].std()
    mae_mean  = -scores['test_mae'].mean()
    r2_mean   = scores['test_r2'].mean()
    
    leaderboard_rows.append({
        'model':         name,
        'cv_rmse_mean':  round(rmse_mean, 4),
        'cv_rmse_std':   round(rmse_std, 4),
        'cv_mae_mean':   round(mae_mean, 4),
        'cv_r2_mean':    round(r2_mean, 4),
        'test_score':    None,
        'note':          'CV',
    })
    
    # Save first-fold fitted model + validation split for SHAP
    first_split = list(kf.split(X_train))[0]
    _, val_idx = first_split
    X_val_shap = X_train.iloc[val_idx]
    fitted_models[name] = scores['estimator'][0]
    shap_data[name] = X_val_shap
    
    print(f"{name:30s}  RMSE={rmse_mean:.4f} ±{rmse_std:.4f}  MAE={mae_mean:.4f}  R²={r2_mean:.4f}")

leaderboard = pd.DataFrame(leaderboard_rows).sort_values('cv_rmse_mean').reset_index(drop=True)
print("\n--- LEADERBOARD ---")
display(leaderboard[['model','cv_rmse_mean','cv_rmse_std','cv_mae_mean','cv_r2_mean']])

In [ ]:
models_dir = PROJECT_ROOT / 'models'
models_dir.mkdir(exist_ok=True)
leaderboard_path = models_dir / 'leaderboard.csv'

if leaderboard_path.exists():
    existing = pd.read_csv(leaderboard_path)
    leaderboard_all = pd.concat([existing, leaderboard], ignore_index=True)
else:
    leaderboard_all = leaderboard.copy()

leaderboard_all.to_csv(leaderboard_path, index=False)
print(f"Leaderboard saved to {leaderboard_path.relative_to(PROJECT_ROOT)}")
print(f"Total rows in leaderboard: {len(leaderboard_all)}")

## Step 4: SHAP Explainability

SHAP values computed for every model on the **validation fold of the first CV split** (not the full training set — faster and representative).

Explainer type per model:
- **Tree-based** (RF, XGBoost, LGBM): TreeExplainer
- **Linear** (Ridge, Lasso, LinearRegression): LinearExplainer
- **Other** (Dummy): skipped — no meaningful explanation

Plots saved to `reports/shap_{model_name}.png` for use in Phase 9 modelling report (MOD-08).

In [ ]:
reports_dir = PROJECT_ROOT / 'reports'
reports_dir.mkdir(exist_ok=True)

TREE_MODELS = ('RandomForestRegressor', 'XGBRegressor', 'LGBMRegressor')
LINEAR_MODELS = ('LinearRegression', 'Ridge', 'Lasso')
SKIP_MODELS = ('DummyRegressor(mean)',)

for name, fitted_model in fitted_models.items():
    if name in SKIP_MODELS:
        print(f"{name}: skipped (no meaningful SHAP explanation)")
        continue
    
    X_shap = shap_data[name]
    
    try:
        if name in TREE_MODELS:
            explainer = shap.TreeExplainer(fitted_model)
            shap_values = explainer.shap_values(X_shap)
        elif name in LINEAR_MODELS:
            explainer = shap.LinearExplainer(fitted_model, X_shap)
            shap_values = explainer.shap_values(X_shap)
        else:
            background = shap.sample(X_shap, min(100, len(X_shap)))
            explainer = shap.KernelExplainer(fitted_model.predict, background)
            shap_values = explainer.shap_values(X_shap, nsamples=100)
        
        plt.figure(figsize=(8, 5))
        shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False,
                          max_display=10, title=f'SHAP — {name}')
        safe_name = name.replace('(', '').replace(')', '').replace(' ', '_')
        plot_path = reports_dir / f'shap_{safe_name}.png'
        plt.savefig(plot_path, bbox_inches='tight', dpi=100)
        plt.show()
        print(f"Saved: {plot_path.relative_to(PROJECT_ROOT)}")
    
    except Exception as e:
        print(f"{name}: SHAP failed — {e}")

## Step 5: Optuna Hyperparameter Tuning

Tuning the **top 3 non-Dummy models** from the leaderboard. 30 trials each.

Objective: minimize RMSE on 5-fold CV.

Tuned results are appended to the leaderboard with a `(tuned)` suffix.

In [ ]:
# Top 3 non-Dummy models from leaderboard
top3 = leaderboard[~leaderboard['model'].str.contains('Dummy')].head(3)['model'].tolist()
print(f"Top 3 models selected for Optuna tuning: {top3}")

In [ ]:
def make_objective(model_name, X_tr, y_tr, kf, seed):
    def objective(trial):
        if 'RandomForest' in model_name:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 15),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            }
            model = RandomForestRegressor(random_state=seed, **params)
        elif 'XGB' in model_name:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            }
            model = xgb.XGBRegressor(random_state=seed, verbosity=0, **params)
        elif 'LGBM' in model_name:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            }
            model = lgb.LGBMRegressor(random_state=seed, verbose=-1, **params)
        elif 'Ridge' in model_name:
            params = {'alpha': trial.suggest_float('alpha', 0.01, 100.0, log=True)}
            model = Ridge(**params)
        elif 'Lasso' in model_name:
            params = {'alpha': trial.suggest_float('alpha', 0.001, 10.0, log=True)}
            model = Lasso(max_iter=10000, **params)
        else:
            model = LinearRegression()
        
        scores = cross_validate(model, X_tr, y_tr, cv=kf,
                                scoring='neg_root_mean_squared_error')
        return -scores['test_score'].mean()
    
    return objective

In [ ]:
tuned_rows = []
tuned_models = {}

for model_name in top3:
    print(f"\nTuning {model_name} (30 trials)...")
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(make_objective(model_name, X_train, y_train, kf, SEED),
                   n_trials=30, show_progress_bar=False)
    
    best_params = study.best_params
    best_rmse = study.best_value
    print(f"  Best RMSE: {best_rmse:.4f}  |  Params: {best_params}")
    
    # Refit on full training set with best params
    if 'RandomForest' in model_name:
        best_model = RandomForestRegressor(random_state=SEED, **best_params)
    elif 'XGB' in model_name:
        best_model = xgb.XGBRegressor(random_state=SEED, verbosity=0, **best_params)
    elif 'LGBM' in model_name:
        best_model = lgb.LGBMRegressor(random_state=SEED, verbose=-1, **best_params)
    elif 'Ridge' in model_name:
        best_model = Ridge(**best_params)
    elif 'Lasso' in model_name:
        best_model = Lasso(max_iter=10000, **best_params)
    else:
        best_model = LinearRegression()
    
    best_model.fit(X_train, y_train)
    tuned_models[model_name] = best_model
    
    tuned_rows.append({
        'model':        f'{model_name} (tuned)',
        'cv_rmse_mean': round(best_rmse, 4),
        'cv_rmse_std':  None,
        'cv_mae_mean':  None,
        'cv_r2_mean':   None,
        'test_score':   None,
        'note':         'Optuna-tuned',
    })

tuned_df = pd.DataFrame(tuned_rows)
leaderboard = pd.concat([leaderboard, tuned_df], ignore_index=True).sort_values('cv_rmse_mean').reset_index(drop=True)
print("\n--- UPDATED LEADERBOARD (with tuned) ---")
display(leaderboard[['model','cv_rmse_mean','note']])

## Step 6: Test Set Reveal ★

The best tuned model is now evaluated on the **held-out test set** (20% of data, sealed since Step 1).

This is the **only time the test set is used**. This score estimates real-world performance.

In [ ]:
# Best tuned model = lowest cv_rmse_mean among tuned models
best_tuned_name = tuned_df.sort_values('cv_rmse_mean').iloc[0]['model']
base_name = best_tuned_name.replace(' (tuned)', '')
best_model = tuned_models[base_name]

y_pred_test = best_model.predict(X_test)
test_rmse = mean_squared_error(y_test, y_pred_test, squared=False)
test_mae  = mean_absolute_error(y_test, y_pred_test)
test_r2   = r2_score(y_test, y_pred_test)

print(f"Best model: {best_tuned_name}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE:  {test_mae:.4f}")
print(f"Test R²:   {test_r2:.4f}")

# Append test reveal row to leaderboard
test_row = pd.DataFrame([{
    'model':        f'{best_tuned_name} ★ TEST',
    'cv_rmse_mean': test_rmse,
    'cv_rmse_std':  None,
    'cv_mae_mean':  test_mae,
    'cv_r2_mean':   test_r2,
    'test_score':   test_rmse,
    'note':         'TEST SET — final evaluation',
}])
leaderboard = pd.concat([leaderboard, test_row], ignore_index=True)
display(leaderboard[['model','cv_rmse_mean','cv_mae_mean','cv_r2_mean','note']])

In [ ]:
# Save best model — MOD-10
model_path = models_dir / 'best_model.pkl'
joblib.dump(best_model, model_path)
print(f"Best model saved: {model_path.relative_to(PROJECT_ROOT)}")

# Resolve notebook version from filename (set at copy time by execute-phase.md)
notebook_version = 'v1'  # execute-phase.md sets this; update if running manually

metadata = {
    'model_name':       best_tuned_name,
    'problem_type':     problem_type,
    'cv_metric':        'rmse',
    'cv_score_mean':    float(tuned_df.sort_values('cv_rmse_mean').iloc[0]['cv_rmse_mean']),
    'cv_score_std':     None,
    'test_score':       round(float(test_rmse), 4),
    'feature_names':    X.columns.tolist(),
    'hyperparameters':  best_model.get_params(),
    'training_date':    str(date.today()),
    'notebook_version': notebook_version,
}

metadata_path = models_dir / 'model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"Metadata saved: {metadata_path.relative_to(PROJECT_ROOT)}")
print(json.dumps(metadata, indent=2, default=str))

In [ ]:
leaderboard_all_final = pd.read_csv(leaderboard_path) if leaderboard_path.exists() else pd.DataFrame()
leaderboard_all_final = pd.concat(
    [leaderboard_all_final, leaderboard[leaderboard['note'].isin(['Optuna-tuned', 'TEST SET — final evaluation'])]],
    ignore_index=True
)
leaderboard_all_final.to_csv(leaderboard_path, index=False)
print(f"Final leaderboard ({len(leaderboard_all_final)} rows) saved to {leaderboard_path.relative_to(PROJECT_ROOT)}")

## Interpretation & Recommendations

*This section is written by Claude after notebook execution (Decision 8).*

**Instructions for execute-phase.md:** After running this notebook, read `models/leaderboard.csv` and the SHAP output files in `reports/`, then replace this cell with:
1. The top model finding (which model won and by how much vs. baseline)
2. Any anomalies (e.g., low baseline gap suggesting features are weak, or high CV std suggesting overfitting)
3. Suggested next steps (e.g., deeper Optuna tuning, feature engineering, or "results look solid — proceed to reports")

*To create a new modelling iteration with a different focus: run `/doml-iterate-model "your direction here"`*

---

## Caveats & Limitations

- **Correlation is not causation.** Model feature importances reflect statistical association, not causal relationships.
- CV metrics are estimates of generalization error, not guarantees. Performance on new data may differ.
- The test score (Step 6) is the single most reliable estimate of out-of-sample performance — but it is still an estimate on one specific 20% split.
- Optuna tunes hyperparameters on CV score; the optimal hyperparameters for CV may not be optimal for the full training set.
- Models were not evaluated for fairness or bias across demographic groups. If the target variable or features correlate with protected attributes, additional fairness analysis is required.